In [60]:
import json
from collections import Counter

file_path = "data/techmap_jobs_sample_30000.jsonl"

field_counter = Counter()
sample_records = []

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= 1000:
            break

        record = json.loads(line)
        field_counter.update(record.keys())

        if i < 3:
            sample_records.append(record)

print("Records inspected:", i + 1)

print("\nMost common fields:")
for field, count in field_counter.most_common():
    print(field, count)

print("\nSample record keys:")
print(sample_records[0].keys())

print("\nFirst sample record:")
sample = sample_records[0]
for k, v in sample.items():
    print(f"{k}: {str(v)[:300]}")

Records inspected: 1001

Most common fields:
_id 1000
sourceCC 1000
source 1000
idInSource 1000
locationID 1000
companyID 1000
text 1000
html 1000
json 1000
locale 1000
position 1000
orgAddress 1000
orgCompany 1000
name 1000
url 1000
dateScraped 1000
dateUploaded 1000
dateCreated 1000
orgTags 1000
contact 635

Sample record keys:
dict_keys(['_id', 'sourceCC', 'source', 'idInSource', 'locationID', 'companyID', 'text', 'html', 'json', 'locale', 'position', 'contact', 'orgAddress', 'orgCompany', 'name', 'url', 'dateScraped', 'dateUploaded', 'dateCreated', 'orgTags'])

First sample record:
_id: {'$oid': '61305cf77e6d8f148767bf29'}
sourceCC: us
source: aarp_us
idInSource: c095b8d8081fe810c166de5a88913585b
locationID: {'$oid': '5f8e5f506692263c6a1cb213'}
companyID: {'$oid': '610786abbd310d02bec97111'}
text: The Brink's name is a promise to respect the trust we've earned in over 150 years in business. Every employee honors that promise by offering the highest levels of service and support to 

In [61]:
total = sum(weights.values())

weights = {
    k: v/total
    for k,v in weights.items()
}

weights

{'semantic': 0.1888929870573626,
 'skill': 0.2204460499823843,
 'role': 0.5528823655487805,
 'location': 0.03777859741147253}

In [4]:
import sys
!{sys.executable} -m pip install pandas tqdm

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [62]:
import pandas as pd
from tqdm import tqdm

print("pandas works")

pandas works


In [65]:
import json
import pandas as pd
import hashlib
from tqdm import tqdm

file_path = "data/techmap_jobs_sample_30000.jsonl"
output_path = "data/jobpilot_jobs_snapshot.csv"

MAX_RECORDS = 25000

def safe_get_date(value):
    if isinstance(value, dict) and "$date" in value:
        return value["$date"]
    return value

def extract_company(value):
    if isinstance(value, dict):
        return (
            value.get("name")
            or value.get("companyName")
            or value.get("title")
            or value.get("idInSource")
            or ""
        )
    return value or ""

def extract_location(value):
    if isinstance(value, dict):
        return (
            value.get("addressLine")
            or value.get("city")
            or value.get("region")
            or value.get("country")
            or ""
        )
    return value or ""

def make_hash(title, company, location, description):
    raw = f"{title}|{company}|{location}|{description[:500]}".lower().strip()
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

rows = []
seen_hashes = set()

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in tqdm(f, total=MAX_RECORDS):
        if len(rows) >= MAX_RECORDS:
            break

        try:
            record = json.loads(line)
        except:
            continue

        title = record.get("name", "")
        company = extract_company(record.get("orgCompany", ""))
        location = extract_location(record.get("orgAddress", ""))
        description = record.get("text", "")
        job_url = record.get("url", "")
        country = record.get("sourceCC", "")
        date_posted = safe_get_date(record.get("dateCreated", ""))

        if not title or not description:
            continue

        if not company:
            company = "Unknown Company"

        dedup_key = make_hash(title, company, location, description)

        if dedup_key in seen_hashes:
            continue

        seen_hashes.add(dedup_key)

        rows.append({
            "job_id": len(rows) + 1,
            "title": title,
            "company": company,
            "location": location,
            "country": country,
            "description": description,
            "job_url": job_url,
            "date_posted": date_posted,
            "salary_min": None,
            "salary_max": None,
            "salary_text": "",
            "source": "Techmap Kaggle",
            "dedup_key": dedup_key
        })

jobs_df = pd.DataFrame(rows)

print("Clean records:", len(jobs_df))
print(jobs_df[["title", "company", "location", "country"]].head(10))
print(jobs_df.info())

jobs_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

27439it [00:07, 3902.53it/s]                           


Clean records: 25000
                                      title                          company  \
0                     Account Executive Sr.                           Brinks   
1                   Global Client Executive                   Konica Minolta   
2     Retirement Plan Sales Support Analyst                           Sentry   
3             Digital Channel Product Owner                         Vanguard   
4  Account Coordinator - HUB International-  Skills For Chicagoland's Future   
5                        Customer Suppt Rep                   Konica Minolta   
6               Project Specialist ChemCare                 Univar Solutions   
7                         Scale Coordinator                   Kelly Services   
8                    Engagement Coordinator                       Home Goods   
9                             Brand Warrior                            T-ROC   

             location country  
0        New York, NY      us  
1   San Francisco, CA      us  
2 

In [66]:
import pandas as pd

snapshot_path = "data/jobpilot_jobs_snapshot.csv"

jobs_df = pd.read_csv(snapshot_path)

print("Rows:", len(jobs_df))
print("Unique dedup keys:", jobs_df["dedup_key"].nunique())
print("Duplicate dedup keys:", jobs_df["dedup_key"].duplicated().sum())

print("\nMissing values:")
print(jobs_df[["title", "company", "location", "description", "job_url"]].isna().sum())

print("\nTop companies:")
print(jobs_df["company"].value_counts().head(10))

print("\nTop locations:")
print(jobs_df["location"].value_counts().head(10))

Rows: 25000
Unique dedup keys: 25000
Duplicate dedup keys: 0

Missing values:
title          0
company        0
location       0
description    0
job_url        0
dtype: int64

Top companies:
company
Appcastenterprise                 862
Varsity Tutors                    605
siehe Beschreibung                339
Marathon Staffing                 223
Anderson Merchandisers, L.L.C.    190
Army National Guard               174
Macy's                            155
Advantage Solutions               141
Swing Education                   137
Adecco                            124
Name: count, dtype: int64

Top locations:
location
London , South East England    347
Sydney                         345
Berlin                         322
New York, NY                   308
Melbourne                      303
Auckland                       231
Brisbane                       209
China                          208
Shanghai, China                206
Hamburg                        175
Name: count, dtype:

In [1]:
import sys
!{sys.executable} -m pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 343 kB 403 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import pypdf
print("pypdf works")

pypdf works


In [67]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """
    Extract all text from a PDF resume.
    """

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [68]:
resume_path = "Naseem_Ali_Resume.pdf"

resume_text = extract_text_from_pdf(resume_path)

print(resume_text[:3000])

 
Naseem  Ali  American  Canyon,  CA  |  (707)  299-8703  |  naseem96ali@gmail.com  |  www.linkedin.com/in/naseemali22  MSBA  student  with  a  STEM  and  engineering  background,  building  experience  in  data  analytics  through  SQL,  
Python,
 
R,
 
and
 
applied
 
projects.
 
Aspiring
 
data
 
analyst
 
leveraging
 
SQL,
 
Python,
 
and
 
machine
 
learning
 
to
 
support
 
business,
 
operations,
 
and
 
marketing
 
analytics.
  EDUCATION   Master  of  Science  in  Business  Analytics  UC  Davis  |  August  2025  -  July  2026   
Bachelor  of  Science  in  Bioengineering  UC  San  Diego,  San  Diego  |  September  2019  -  June  2022  GPA:  3.25  SKILLS  Programming:  Python,  SQL  (MySQL),  R,  Matlab  Data  &  Tools:  Databricks,  Power  BI,  Excel,  Jira,  Asana,  Outlook,  Teams  Analysis  &  Modeling:  Regression,  Optimization,  A/B  Testing,  Data  Cleaning,  Visualization  Engineering  &  Design:  SolidWorks,  AutoCAD   WORK  EXPERIENCE  Data  Analytics  Practicum  Inter

In [69]:
import re

SKILL_KEYWORDS = [
    # Programming / analytics
    "python", "r", "sql", "pandas", "numpy", "scikit-learn", "sklearn",
    "matplotlib", "seaborn", "tableau", "power bi", "excel",

    # Data engineering / big data
    "pyspark", "spark", "databricks", "mongodb", "mysql", "sqlite",
    "etl", "api", "data pipeline", "data cleaning",

    # Machine learning / statistics
    "machine learning", "logistic regression", "regression", "classification",
    "clustering", "pca", "random forest", "decision tree", "nlp",
    "sentiment analysis", "topic modeling",

    # Business / operations
    "data analytics", "business analytics", "dashboard", "visualization",
    "quality engineering", "process improvement", "cross-functional",
    "root cause analysis", "documentation",

    # Tools / engineering
    "solidworks", "catia", "jira", "git", "github"
]

def extract_skills(text, skill_keywords=SKILL_KEYWORDS):
    """
    Extract skills from resume text using a controlled keyword dictionary.
    """
    text_lower = text.lower()
    matched_skills = []

    for skill in skill_keywords:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, text_lower):
            matched_skills.append(skill)

    return sorted(set(matched_skills))

resume_skills = extract_skills(resume_text)

print("Skills found:")
print(resume_skills)
print("\nNumber of skills found:", len(resume_skills))

Skills found:
['api', 'clustering', 'databricks', 'documentation', 'excel', 'jira', 'mysql', 'nlp', 'pca', 'python', 'r', 'regression', 'solidworks', 'sql', 'sqlite', 'tableau', 'visualization']

Number of skills found: 17


In [70]:
def build_user_profile(
    resume_text,
    skills,
    target_roles=None,
    preferred_locations=None,
    salary_min=None,
    dealbreakers=None
):
    """
    Build a standardized user profile object for retrieval and ranking.
    """

    if target_roles is None:
        target_roles = ["Data Analyst", "Business Analyst", "Analytics Engineer", "Data Scientist"]

    if preferred_locations is None:
        preferred_locations = ["Remote", "Bay Area", "San Francisco", "Oakland", "San Jose"]

    if dealbreakers is None:
        dealbreakers = ["contract only", "unpaid", "senior", "staff", "principal"]

    profile = {
        "resume_text": resume_text,
        "skills": skills,
        "target_roles": target_roles,
        "preferred_locations": preferred_locations,
        "salary_min": salary_min,
        "dealbreakers": dealbreakers
    }

    return profile

user_profile = build_user_profile(
    resume_text=resume_text,
    skills=resume_skills,
    salary_min=80000
)

user_profile

{'resume_text': ' \nNaseem  Ali  American  Canyon,  CA  |  (707)  299-8703  |  naseem96ali@gmail.com  |  www.linkedin.com/in/naseemali22  MSBA  student  with  a  STEM  and  engineering  background,  building  experience  in  data  analytics  through  SQL,  \nPython,\n \nR,\n \nand\n \napplied\n \nprojects.\n \nAspiring\n \ndata\n \nanalyst\n \nleveraging\n \nSQL,\n \nPython,\n \nand\n \nmachine\n \nlearning\n \nto\n \nsupport\n \nbusiness,\n \noperations,\n \nand\n \nmarketing\n \nanalytics.\n  EDUCATION   Master  of  Science  in  Business  Analytics  UC  Davis  |  August  2025  -  July  2026   \nBachelor  of  Science  in  Bioengineering  UC  San  Diego,  San  Diego  |  September  2019  -  June  2022  GPA:  3.25  SKILLS  Programming:  Python,  SQL  (MySQL),  R,  Matlab  Data  &  Tools:  Databricks,  Power  BI,  Excel,  Jira,  Asana,  Outlook,  Teams  Analysis  &  Modeling:  Regression,  Optimization,  A/B  Testing,  Data  Cleaning,  Visualization  Engineering  &  Design:  SolidWorks,  

In [71]:
import re

def infer_career_level(resume_text):

    text = resume_text.lower()

    senior_terms = [
        "senior",
        "sr.",
        "staff",
        "principal",
        "lead",
        "manager",
        "director",
        "vice president",
        "vp"
    ]

    junior_terms = [
        "intern",
        "internship",
        "student",
        "graduate",
        "junior",
        "jr."
    ]

    # Strong senior signal
    for term in senior_terms:
        if term in text:
            return "senior"

    # Strong junior signal
    for term in junior_terms:
        if term in text:
            return "early"

    # Estimate based on years appearing in resume
    years = re.findall(r"(20\d{2})", resume_text)

    if len(years) >= 2:
        try:
            years = sorted([int(y) for y in years])

            experience_years = years[-1] - years[0]

            if experience_years <= 2:
                return "early"
            elif experience_years <= 7:
                return "mid"
            else:
                return "senior"

        except:
            pass

    return "early"

In [72]:
user_profile = build_user_profile(
    resume_text=resume_text,
    skills=resume_skills,
    salary_min=80000
)

user_profile["career_level"] = infer_career_level(
    resume_text
)

print("Career Level:", user_profile["career_level"])

user_profile

Career Level: early


{'resume_text': ' \nNaseem  Ali  American  Canyon,  CA  |  (707)  299-8703  |  naseem96ali@gmail.com  |  www.linkedin.com/in/naseemali22  MSBA  student  with  a  STEM  and  engineering  background,  building  experience  in  data  analytics  through  SQL,  \nPython,\n \nR,\n \nand\n \napplied\n \nprojects.\n \nAspiring\n \ndata\n \nanalyst\n \nleveraging\n \nSQL,\n \nPython,\n \nand\n \nmachine\n \nlearning\n \nto\n \nsupport\n \nbusiness,\n \noperations,\n \nand\n \nmarketing\n \nanalytics.\n  EDUCATION   Master  of  Science  in  Business  Analytics  UC  Davis  |  August  2025  -  July  2026   \nBachelor  of  Science  in  Bioengineering  UC  San  Diego,  San  Diego  |  September  2019  -  June  2022  GPA:  3.25  SKILLS  Programming:  Python,  SQL  (MySQL),  R,  Matlab  Data  &  Tools:  Databricks,  Power  BI,  Excel,  Jira,  Asana,  Outlook,  Teams  Analysis  &  Modeling:  Regression,  Optimization,  A/B  Testing,  Data  Cleaning,  Visualization  Engineering  &  Design:  SolidWorks,  

In [14]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 488 kB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 8.0 MB 136 kB/s eta 0:00:011
     |████████████████████████████████| 39.4 MB 31 kB/s s eta 0:00:01    |█▊                              | 2.1 MB 31.3 MB/s eta 0:00:02
     |████████████████████████████████| 12.0 MB 7.3 MB/s eta 0:00:01    |█████▉                          | 2.2 MB 7.3 MB/s eta 0:00:02
     |████████████████████████████████| 12.1 MB 23.2 MB/s eta 0:00:01    |█████████████▉                  | 5.2 MB 23.2 MB/s eta 0:00:01
     |████████████████████████████████| 625 kB 9.8 MB/s eta 0:00:01
     |████████████████████████████████| 150.8 MB 64 kB/s s eta 0:00:01  |█▉                              | 8.5 MB 4.4 MB/s eta 0:00:33     |██▊                             | 13.0 MB 4.4 MB/s eta 0:00:32     |████                            | 19.0 MB 4.4 MB/s eta 0:00:31     |█████████████▎                  

In [73]:
from sentence_transformers import SentenceTransformer
import faiss

print("SentenceTransformers and FAISS work")

SentenceTransformers and FAISS work


In [4]:
import sys

!{sys.executable} -m pip install "numpy<2" --force-reinstall

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp39-cp39-macosx_10_9_x86_64.whl (20.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [20]:
import numpy as np
print(np.__version__)

from sentence_transformers import SentenceTransformer
import faiss

print("SentenceTransformers and FAISS work")

1.26.4
SentenceTransformers and FAISS work


In [74]:
jobs_df["embedding_text"] = (
    "Title: " + jobs_df["title"].fillna("") + ". " +
    "Company: " + jobs_df["company"].fillna("") + ". " +
    "Location: " + jobs_df["location"].fillna("") + ". " +
    "Description: " + jobs_df["description"].fillna("")
)

print(jobs_df["embedding_text"].iloc[0][:1000])

Title: Account Executive Sr.. Company: Brinks. Location: New York, NY. Description: The Brink's name is a promise to respect the trust we've earned in over 150 years in business. Every employee honors that promise by offering the highest levels of service and support to our customers. We take pride in our work, and we share a passion about our future. Learn why so many people have made the choice to join our team - and stay here. Job Title Account Executive - Sr. Job Description Brink's Global Services is a division of Brink's Inc, the world's premier provider of secure logistics and security solutions in more than 122 countries across 5 continents. Brink's Global Services specializes in the secure transportation and handling of valuable goods throughout the logistic value chain, from raw materials and components to finished products within the mining, banknote, precious metal, jewelry, security, art and pharmaceutical industries The company has a proud history of providing growth and 

In [75]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

job_texts = jobs_df["embedding_text"].tolist()

job_embeddings = model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", job_embeddings.shape)

Batches: 100%|██████████| 391/391 [34:36<00:00,  5.31s/it]


Embedding shape: (25000, 384)


In [76]:
#Build FAISS Index

import faiss

embedding_dim = job_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(job_embeddings)

print("Number of vectors in FAISS index:", index.ntotal)

Number of vectors in FAISS index: 25000


In [77]:
#create user profile text

def build_profile_text(user_profile):
    profile_text = ""

    profile_text += "Target roles: " + ", ".join(user_profile["target_roles"]) + ". "
    profile_text += "Skills: " + ", ".join(user_profile["skills"]) + ". "
    profile_text += "Preferred locations: " + ", ".join(user_profile["preferred_locations"]) + ". "
    profile_text += "Resume summary: " + user_profile["resume_text"][:3000]

    return profile_text

profile_text = build_profile_text(user_profile)

print(profile_text[:1000])

Target roles: Data Analyst, Business Analyst, Analytics Engineer, Data Scientist. Skills: api, clustering, databricks, documentation, excel, jira, mysql, nlp, pca, python, r, regression, solidworks, sql, sqlite, tableau, visualization. Preferred locations: Remote, Bay Area, San Francisco, Oakland, San Jose. Resume summary:  
Naseem  Ali  American  Canyon,  CA  |  (707)  299-8703  |  naseem96ali@gmail.com  |  www.linkedin.com/in/naseemali22  MSBA  student  with  a  STEM  and  engineering  background,  building  experience  in  data  analytics  through  SQL,  
Python,
 
R,
 
and
 
applied
 
projects.
 
Aspiring
 
data
 
analyst
 
leveraging
 
SQL,
 
Python,
 
and
 
machine
 
learning
 
to
 
support
 
business,
 
operations,
 
and
 
marketing
 
analytics.
  EDUCATION   Master  of  Science  in  Business  Analytics  UC  Davis  |  August  2025  -  July  2026   
Bachelor  of  Science  in  Bioengineering  UC  San  Diego,  San  Diego  |  September  2019  -  June  2022  GPA:  3.25  SKILLS  Progr

In [78]:
#retrieve top jobs

def retrieve_jobs(profile_text, model, index, jobs_df, top_k=100):
    profile_embedding = model.encode(
        [profile_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(profile_embedding, top_k)

    results = jobs_df.iloc[indices[0]].copy()
    results["semantic_similarity"] = scores[0]

    return results.reset_index(drop=True)

retrieved_jobs = retrieve_jobs(
    profile_text=profile_text,
    model=model,
    index=index,
    jobs_df=jobs_df,
    top_k=100
)

retrieved_jobs[[
    "job_id",
    "title",
    "company",
    "location",
    "semantic_similarity",
    "job_url"
]].head(10)

,job_id,title,company,location,semantic_similarity,job_url
0,16629,DATA SCIENTIST - INTERMEDIATE,"Judge Group, Inc.","Ladue, MO",0.763356,https://www.dice.com/jobs/detail/edccbdb22f0a2...
1,17783,Data Analytics Engineer,NetPace,Remote,0.748257,https://www.dice.com/jobs/detail/8c815e3146978...
2,23676,Junior Data Scientist,Michael Page,"Slough,Slough",0.732668,https://www.britishjobs.co.uk/job/121000000000...
3,17767,Senior Analytics Specialist,Finite920,Auckland,0.720095,https://www.seek.co.nz/job/53721106
4,23679,Data Scientist,Michael Page,"Slough,Slough",0.719323,https://www.britishjobs.co.uk/job/121000000000...
5,12265,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.712578,https://www.reed.co.uk/jobs/graduate-data-anal...
6,12339,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.710085,https://www.reed.co.uk/jobs/graduate-data-anal...
7,12384,Senior Data Scientist,Mexa Solutions LTD,"Southampton , Hampshire",0.706509,https://www.reed.co.uk/jobs/senior-data-scient...
8,13023,Data Analyst,Millennium Consulting,"Providence, RI",0.700256,https://www.dice.com/jobs/detail/feac79d0c0a27...
9,12851,Data Engineer,Collabera,"Charlotte, NC",0.698785,https://www.dice.com/jobs/detail/e67613fe7898a...


In [79]:
#save embeddings and faiss to prevent waiting again if kernel restarts in future

import numpy as np
import faiss

np.save(
    "artifacts/job_embeddings.npy",
    job_embeddings
)

faiss.write_index(
    index,
    "artifacts/job_faiss.index"
)

jobs_df.to_csv(
    "data/jobpilot_jobs_with_embedding_text.csv",
    index=False
)

print("Saved embeddings, FAISS index, and jobs dataframe.")

Saved embeddings, FAISS index, and jobs dataframe.


In [80]:
#run retrieval 

retrieved_jobs = retrieve_jobs(
    profile_text=profile_text,
    model=model,
    index=index,
    jobs_df=jobs_df,
    top_k=100
)

retrieved_jobs[[
    "job_id",
    "title",
    "company",
    "location",
    "semantic_similarity",
    "job_url"
]].head(10)

,job_id,title,company,location,semantic_similarity,job_url
0,16629,DATA SCIENTIST - INTERMEDIATE,"Judge Group, Inc.","Ladue, MO",0.763356,https://www.dice.com/jobs/detail/edccbdb22f0a2...
1,17783,Data Analytics Engineer,NetPace,Remote,0.748257,https://www.dice.com/jobs/detail/8c815e3146978...
2,23676,Junior Data Scientist,Michael Page,"Slough,Slough",0.732668,https://www.britishjobs.co.uk/job/121000000000...
3,17767,Senior Analytics Specialist,Finite920,Auckland,0.720095,https://www.seek.co.nz/job/53721106
4,23679,Data Scientist,Michael Page,"Slough,Slough",0.719323,https://www.britishjobs.co.uk/job/121000000000...
5,12265,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.712578,https://www.reed.co.uk/jobs/graduate-data-anal...
6,12339,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.710085,https://www.reed.co.uk/jobs/graduate-data-anal...
7,12384,Senior Data Scientist,Mexa Solutions LTD,"Southampton , Hampshire",0.706509,https://www.reed.co.uk/jobs/senior-data-scient...
8,13023,Data Analyst,Millennium Consulting,"Providence, RI",0.700256,https://www.dice.com/jobs/detail/feac79d0c0a27...
9,12851,Data Engineer,Collabera,"Charlotte, NC",0.698785,https://www.dice.com/jobs/detail/e67613fe7898a...


In [81]:
#Apply hard filter

def apply_hard_filters(candidates_df, user_profile):
    """
    Remove jobs that violate seniority or dealbreaker constraints
    based on the user's career level.
    """

    filtered = candidates_df.copy()

    career_level = user_profile.get("career_level", "early").lower()

    senior_terms = [
        "senior", "sr.", "sr ", "staff", "principal",
        "lead", "manager", "director", "head of", "vp"
    ]

    junior_terms = [
        "junior", "jr.", "jr ", "entry level",
        "graduate", "intern", "internship"
    ]

    contract_terms = [
    "contract only",
    "unpaid",
    "temporary position",
    "temporary role",
    "contract position",
    "contract role"
]

    def violates_filter(row):
        title = str(row.get("title", "")).lower()
        description = str(row.get("description", "")).lower()

        # Always remove unpaid/temp/contract-only roles if present
        combined_text = title + " " + description
        for term in contract_terms:
            if term in combined_text:
                return True

        # Early-career candidates should not receive senior roles
        if career_level in ["student", "early", "new_grad"]:
            for term in senior_terms:
                if term in title:
                    return True

        # Senior candidates should not receive junior roles
        if career_level in ["senior", "experienced"]:
            for term in junior_terms:
                if term in title:
                    return True

        return False

    filtered["violates_dealbreaker"] = filtered.apply(violates_filter, axis=1)

    removed_jobs = filtered[filtered["violates_dealbreaker"] == True]
    kept_jobs = filtered[filtered["violates_dealbreaker"] == False].copy()

    return kept_jobs.reset_index(drop=True), removed_jobs.reset_index(drop=True)

In [82]:
filtered_jobs, removed_jobs = apply_hard_filters(
    retrieved_jobs,
    user_profile
)

print("Career level:", user_profile["career_level"])
print("Retrieved jobs:", len(retrieved_jobs))
print("Kept after hard filters:", len(filtered_jobs))
print("Removed by hard filters:", len(removed_jobs))

removed_jobs[["title", "company"]].head(20)

Career level: early
Retrieved jobs: 100
Kept after hard filters: 69
Removed by hard filters: 31


,title,company
0,Senior Analytics Specialist,Finite920
1,Senior Data Scientist,Mexa Solutions LTD
2,Data Analyst,The Carrera Agency
3,"Sr. Data Scientist, GTM Enablement",DataRobot
4,Senior Data Analytics Engineerr,Jobot
5,Sr. Software Engineer (Machine Learning Platform),Zscaler
6,Direct Client Requirement Sr Data Engineer (10...,Kairos
7,SR Business Intelligence (BI) Analyst,Robert Half
8,"Sr Analyst, Finance",TalentBurst
9,Senior Product Analyst (Entertainment),Harnham


In [83]:
import re
import numpy as np

def normalize_score(series):
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        return np.ones(len(series))

    return (series - min_val) / (max_val - min_val)


def compute_skill_match(job_text, user_skills):
    job_text = str(job_text).lower()
    matched = []

    for skill in user_skills:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, job_text):
            matched.append(skill)

    if len(user_skills) == 0:
        return 0, matched

    return len(matched) / len(user_skills), matched


def compute_role_match(title, target_roles):
    title = str(title).lower()

    for role in target_roles:
        role_lower = role.lower()

        if role_lower in title:
            return 1.0

        role_words = role_lower.split()
        if all(word in title for word in role_words):
            return 1.0

    return 0.0


def compute_location_match(location, preferred_locations):
    location = str(location).lower()

    for preferred in preferred_locations:
        preferred = preferred.lower()

        if preferred in location:
            return 1.0

        if preferred == "bay area" and any(
            city in location for city in ["san francisco", "oakland", "san jose", "berkeley", "palo alto"]
        ):
            return 1.0

        if preferred == "california" and any(
            ca in location for ca in ["ca", "california", "san francisco", "san jose", "oakland", "los angeles", "san diego"]
        ):
            return 1.0

    return 0.0


def rank_jobs(filtered_jobs, user_profile):
    ranked = filtered_jobs.copy()

    ranked["combined_job_text"] = (
        ranked["title"].fillna("") + " " +
        ranked["description"].fillna("")
    )

    skill_results = ranked["combined_job_text"].apply(
        lambda text: compute_skill_match(text, user_profile["skills"])
    )

    ranked["skill_match_score"] = skill_results.apply(lambda x: x[0])
    ranked["matched_skills"] = skill_results.apply(lambda x: x[1])

    ranked["role_match_score"] = ranked["title"].apply(
        lambda title: compute_role_match(title, user_profile["target_roles"])
    )

    ranked["location_match_score"] = ranked["location"].apply(
        lambda location: compute_location_match(location, user_profile["preferred_locations"])
    )

    ranked["semantic_score_norm"] = normalize_score(ranked["semantic_similarity"])

    ranked["final_score"] = (
        0.50 * ranked["semantic_score_norm"] +
        0.25 * ranked["skill_match_score"] +
        0.15 * ranked["role_match_score"] +
        0.10 * ranked["location_match_score"]
    )

    ranked = ranked.sort_values("final_score", ascending=False)

    return ranked.reset_index(drop=True)


ranked_jobs = rank_jobs(filtered_jobs, user_profile)

ranked_jobs[[
    "job_id",
    "title",
    "company",
    "location",
    "semantic_similarity",
    "skill_match_score",
    "role_match_score",
    "location_match_score",
    "final_score",
    "matched_skills"
]].head(10)

,job_id,title,company,location,semantic_similarity,skill_match_score,role_match_score,location_match_score,final_score,matched_skills
0,17783,Data Analytics Engineer,NetPace,Remote,0.748257,0.176471,1.0,1.0,0.751735,"[databricks, python, sql]"
1,16629,DATA SCIENTIST - INTERMEDIATE,"Judge Group, Inc.","Ladue, MO",0.763356,0.117647,1.0,0.0,0.679412,"[python, sql]"
2,23676,Junior Data Scientist,Michael Page,"Slough,Slough",0.732668,0.294118,1.0,0.0,0.637390,"[documentation, python, r, sql, visualization]"
3,23679,Data Scientist,Michael Page,"Slough,Slough",0.719323,0.294118,1.0,0.0,0.599929,"[documentation, python, r, sql, visualization]"
4,12265,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.712578,0.235294,1.0,0.0,0.566289,"[excel, python, r, sql]"
5,12339,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.710085,0.235294,1.0,0.0,0.559293,"[excel, python, r, sql]"
6,12951,Data Scientist (Remote),The Execu|Search Group,"Washington, DC",0.697207,0.294118,1.0,0.0,0.537851,"[clustering, mysql, python, r, regression]"
7,13023,Data Analyst,Millennium Consulting,"Providence, RI",0.700256,0.117647,1.0,0.0,0.502291,"[documentation, sql]"
8,13660,Data Analyst Trainee (m/f/d),Bigpoint GmbH,Hamburg,0.676190,0.352941,1.0,0.0,0.493561,"[clustering, python, r, regression, sql, tableau]"
9,12267,Graduate Data Analyst,ITECCO,"Bristol , Avon",0.680695,0.235294,1.0,0.0,0.476796,"[excel, python, r, sql]"


In [84]:
def generate_explanation(job_row, user_profile):

    explanation_parts = []

    # Skills
    matched_skills = job_row["matched_skills"]

    if len(matched_skills) > 0:
        explanation_parts.append(
            f"Matched skills: {', '.join(matched_skills)}"
        )

    # Role alignment
    if job_row["role_match_score"] > 0:
        explanation_parts.append(
            "Matches one of your target roles."
        )

    # Location alignment
    if job_row["location_match_score"] > 0:
        explanation_parts.append(
            "Matches one of your preferred locations."
        )

    # Semantic similarity
    explanation_parts.append(
        f"Semantic similarity score: {job_row['semantic_similarity']:.3f}"
    )

    explanation_parts.append(
        f"Final recommendation score: {job_row['final_score']:.3f}"
    )

    return " | ".join(explanation_parts)

In [85]:
ranked_jobs["explanation"] = ranked_jobs.apply(
    lambda row: generate_explanation(
        row,
        user_profile
    ),
    axis=1
)

ranked_jobs[[
    "title",
    "company",
    "final_score",
    "explanation"
]].head(10)

,title,company,final_score,explanation
0,Data Analytics Engineer,NetPace,0.751735,"Matched skills: databricks, python, sql | Matc..."
1,DATA SCIENTIST - INTERMEDIATE,"Judge Group, Inc.",0.679412,"Matched skills: python, sql | Matches one of y..."
2,Junior Data Scientist,Michael Page,0.637390,"Matched skills: documentation, python, r, sql,..."
3,Data Scientist,Michael Page,0.599929,"Matched skills: documentation, python, r, sql,..."
4,Graduate Data Analyst,ITECCO,0.566289,"Matched skills: excel, python, r, sql | Matche..."
5,Graduate Data Analyst,ITECCO,0.559293,"Matched skills: excel, python, r, sql | Matche..."
6,Data Scientist (Remote),The Execu|Search Group,0.537851,"Matched skills: clustering, mysql, python, r, ..."
7,Data Analyst,Millennium Consulting,0.502291,"Matched skills: documentation, sql | Matches o..."
8,Data Analyst Trainee (m/f/d),Bigpoint GmbH,0.493561,"Matched skills: clustering, python, r, regress..."
9,Graduate Data Analyst,ITECCO,0.476796,"Matched skills: excel, python, r, sql | Matche..."


In [86]:
#Adaptive Learning Feedback

ranking_weights = {
    "semantic": 0.50,
    "skill": 0.25,
    "role": 0.15,
    "location": 0.10
}

def rerank_with_weights(filtered_jobs, user_profile, weights):
    ranked = rank_jobs(filtered_jobs, user_profile).copy()

    ranked["final_score"] = (
        weights["semantic"] * ranked["semantic_score_norm"] +
        weights["skill"] * ranked["skill_match_score"] +
        weights["role"] * ranked["role_match_score"] +
        weights["location"] * ranked["location_match_score"]
    )

    ranked = ranked.sort_values("final_score", ascending=False).reset_index(drop=True)

    ranked["explanation"] = ranked.apply(
        lambda row: generate_explanation(row, user_profile),
        axis=1
    )

    return ranked


def update_weights_from_feedback(weights, feedback_rows):
    """
    feedback_rows should include:
    - feedback: accept, reject, or skip
    - skill_match_score
    - role_match_score
    - location_match_score
    - semantic_score_norm
    """

    new_weights = weights.copy()

    accepted = feedback_rows[feedback_rows["feedback"] == "accept"]
    rejected = feedback_rows[feedback_rows["feedback"] == "reject"]

    if len(accepted) > 0:
        if accepted["skill_match_score"].mean() > feedback_rows["skill_match_score"].mean():
            new_weights["skill"] += 0.05

        if accepted["location_match_score"].mean() > feedback_rows["location_match_score"].mean():
            new_weights["location"] += 0.05

        if accepted["role_match_score"].mean() > feedback_rows["role_match_score"].mean():
            new_weights["role"] += 0.05

    if len(rejected) > 0:
        if rejected["location_match_score"].mean() < feedback_rows["location_match_score"].mean():
            new_weights["location"] += 0.03

        if rejected["skill_match_score"].mean() < feedback_rows["skill_match_score"].mean():
            new_weights["skill"] += 0.03

    total = sum(new_weights.values())

    for key in new_weights:
        new_weights[key] = new_weights[key] / total

    return new_weights

In [87]:
# Simulated feedback on current top 10
feedback_sample = ranked_jobs.head(10).copy()

feedback_sample["feedback"] = [
    "accept",  # Data Analytics Engineer Remote
    "accept",  # Data Scientist Intermediate
    "accept",  # Junior Data Scientist
    "skip",
    "accept",
    "accept",
    "reject",
    "skip",
    "reject",
    "accept"
]

updated_weights = update_weights_from_feedback(
    ranking_weights,
    feedback_sample
)

print("Original weights:")
print(ranking_weights)

print("\nUpdated weights:")
print(updated_weights)

adaptive_ranked_jobs = rerank_with_weights(
    filtered_jobs,
    user_profile,
    updated_weights
)

adaptive_ranked_jobs[[
    "title",
    "company",
    "location",
    "final_score",
    "matched_skills",
    "explanation"
]].head(10)

Original weights:
{'semantic': 0.5, 'skill': 0.25, 'role': 0.15, 'location': 0.1}

Updated weights:
{'semantic': 0.4629629629629629, 'skill': 0.23148148148148145, 'role': 0.13888888888888887, 'location': 0.16666666666666669}


,title,company,location,final_score,matched_skills,explanation
0,Data Analytics Engineer,NetPace,Remote,0.770125,"[databricks, python, sql]","Matched skills: databricks, python, sql | Matc..."
1,DATA SCIENTIST - INTERMEDIATE,"Judge Group, Inc.","Ladue, MO",0.629085,"[python, sql]","Matched skills: python, sql | Matches one of y..."
2,Junior Data Scientist,Michael Page,"Slough,Slough",0.590176,"[documentation, python, r, sql, visualization]","Matched skills: documentation, python, r, sql,..."
3,Data Scientist,Michael Page,"Slough,Slough",0.555490,"[documentation, python, r, sql, visualization]","Matched skills: documentation, python, r, sql,..."
4,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.524342,"[excel, python, r, sql]","Matched skills: excel, python, r, sql | Matche..."
5,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",0.517864,"[excel, python, r, sql]","Matched skills: excel, python, r, sql | Matche..."
6,Data Scientist (Remote),The Execu|Search Group,"Washington, DC",0.498010,"[clustering, mysql, python, r, regression]","Matched skills: clustering, mysql, python, r, ..."
7,Data Analyst,Millennium Consulting,"Providence, RI",0.465085,"[documentation, sql]","Matched skills: documentation, sql | Matches o..."
8,Data Analyst Trainee (m/f/d),Bigpoint GmbH,Hamburg,0.457001,"[clustering, python, r, regression, sql, tableau]","Matched skills: clustering, python, r, regress..."
9,Graduate Data Analyst,ITECCO,"Bristol , Avon",0.441478,"[excel, python, r, sql]","Matched skills: excel, python, r, sql | Matche..."


In [88]:
def evaluate_top_k(ranked_df, k=10):
    top_k = ranked_df.head(k)

    avg_final_score = top_k["final_score"].mean()
    avg_skill_score = top_k["skill_match_score"].mean()
    avg_role_score = top_k["role_match_score"].mean()
    avg_location_score = top_k["location_match_score"].mean()

    return {
        "avg_final_score": avg_final_score,
        "avg_skill_score": avg_skill_score,
        "avg_role_score": avg_role_score,
        "avg_location_score": avg_location_score
    }


before_metrics = evaluate_top_k(ranked_jobs, k=10)
after_metrics = evaluate_top_k(adaptive_ranked_jobs, k=10)

benchmark_df = pd.DataFrame([
    {"stage": "Before feedback", **before_metrics},
    {"stage": "After feedback", **after_metrics}
])

benchmark_df

,stage,avg_final_score,avg_skill_score,avg_role_score,avg_location_score
0,Before feedback,0.580455,0.235294,1.0,0.1
1,After feedback,0.544865,0.235294,1.0,0.1


In [89]:
import sys
!{sys.executable} -m pip install -U google-genai

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [114]:
from dotenv import load_dotenv
import os

load_dotenv("file.env")

print("Gemini key loaded:", os.getenv("GEMINI_API_KEY") is not None)

Gemini key loaded: True


In [115]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Write one sentence confirming that the JobPilot resume generator is connected."
)

print(response.text)

The JobPilot resume generator is successfully connected.


In [91]:
import time

def generate_tailored_resume(
    resume_text,
    selected_job,
    client,
    model_name="gemini-2.5-flash",
    max_retries=3
):
    prompt = f"""
You are an expert resume writer.

Your task is to tailor the candidate's resume to the job below.

IMPORTANT RULES:
- Do NOT invent employers.
- Do NOT invent degrees.
- Do NOT invent dates.
- Do NOT invent projects.
- Only rewrite and emphasize existing experience.
- Improve keyword alignment with the target job.
- Highlight relevant skills from the original resume.
- Keep the resume professional and ATS-friendly.

TARGET JOB:
Title: {selected_job['title']}
Company: {selected_job['company']}
Location: {selected_job['location']}
Job Description: {selected_job['description'][:4000]}

ORIGINAL RESUME:
{resume_text[:6000]}

OUTPUT FORMAT:
1. Professional Summary
2. Key Skills
3. Tailored Experience Bullets
4. Why This Resume Fits The Job
"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))

    return "Resume generation failed because the model was temporarily unavailable. Please try again."

In [92]:
selected_job = ranked_jobs.iloc[0]

print(selected_job["title"])
print(selected_job["company"])

Data Analytics Engineer
NetPace


In [93]:
tailored_resume = generate_tailored_resume(
    resume_text=resume_text,
    selected_job=selected_job,
    client=client
)

print(tailored_resume)

Here is the tailored resume for Naseem Ali, optimized for the Data Analytics Engineer role at NetPace:

---

**Naseem Ali**
American Canyon, CA | (707) 299-8703 | naseem96ali@gmail.com | www.linkedin.com/in/naseemali22

---

### Professional Summary

Aspiring Data Analytics Engineer with a strong STEM and engineering foundation, skilled in leveraging **Python, SQL, and Azure Databricks** to design, build, and optimize robust data pipelines. Proven experience in manipulating, processing, and extracting actionable insights from complex datasets, with capabilities in data modeling, visualization, and cross-functional collaboration. Eager to apply expertise in **Azure cloud environments** and **expert SQL statement writing** to drive data-driven solutions and represent key data insights visually in a meaningful way for NetPace.

---

### Key Skills

*   **Programming & Scripting:** Python, SQL (MySQL), R
*   **Azure Cloud & Big Data Tools:** Azure Databricks, PySpark, Power BI, Tableau, Ex

In [94]:
#Batch analytics

from collections import Counter
import pandas as pd
import re

ANALYTICS_SKILLS = [
    "python", "sql", "r", "excel", "tableau", "power bi",
    "databricks", "spark", "pyspark", "aws", "azure", "gcp",
    "machine learning", "nlp", "regression", "clustering",
    "pandas", "numpy", "scikit-learn", "etl", "api",
    "dashboard", "visualization"
]

def count_skill_mentions(jobs_df, skills=ANALYTICS_SKILLS):
    counts = Counter()

    for text in jobs_df["description"].fillna("").astype(str):
        text_lower = text.lower()

        for skill in skills:
            pattern = r"\b" + re.escape(skill.lower()) + r"\b"
            if re.search(pattern, text_lower):
                counts[skill] += 1

    return pd.DataFrame(
        counts.items(),
        columns=["skill", "job_count"]
    ).sort_values("job_count", ascending=False)


top_skills_df = count_skill_mentions(jobs_df)

top_locations_df = (
    jobs_df["location"]
    .value_counts()
    .reset_index()
    .rename(columns={"location": "job_count", "index": "location"})
)

top_companies_df = (
    jobs_df["company"]
    .value_counts()
    .reset_index()
    .rename(columns={"company": "job_count", "index": "company"})
)

top_titles_df = (
    jobs_df["title"]
    .value_counts()
    .reset_index()
    .rename(columns={"title": "job_count", "index": "title"})
)

salary_summary = {
    "total_jobs": len(jobs_df),
    "jobs_with_salary_min": jobs_df["salary_min"].notna().sum(),
    "jobs_with_salary_max": jobs_df["salary_max"].notna().sum(),
    "jobs_with_salary_text": jobs_df["salary_text"].fillna("").astype(str).str.strip().ne("").sum()
}

print("Top Skills")
display(top_skills_df.head(15))

print("Top Locations")
display(top_locations_df.head(15))

print("Top Companies")
display(top_companies_df.head(15))

print("Top Titles")
display(top_titles_df.head(15))

print("Salary Summary")
salary_summary

Top Skills


,skill,job_count
1,excel,2131
0,r,881
2,sql,761
5,python,502
4,aws,414
12,azure,367
3,api,265
14,machine learning,156
7,tableau,136
6,visualization,118


Top Locations


,job_count,count
0,"London , South East England",347
1,Sydney,345
2,Berlin,322
3,"New York, NY",308
4,Melbourne,303
5,Auckland,231
6,Brisbane,209
7,China,208
8,"Shanghai, China",206
9,Hamburg,175


Top Companies


,job_count,count
0,Appcastenterprise,862
1,Varsity Tutors,605
2,siehe Beschreibung,339
3,Marathon Staffing,223
4,"Anderson Merchandisers, L.L.C.",190
5,Army National Guard,174
6,Macy's,155
7,Advantage Solutions,141
8,Swing Education,137
9,Adecco,124


Top Titles


,job_count,count
0,Substitute Teacher - Multi-District Opportunity,82
1,Registered Nurse RGN - Care Home,67
2,Speech-Language Pathologist (SLP) - Skilled Nu...,64
3,Registered Nurse RGN - Bank - Care Home,54
4,Retail Betting Assistant,49
5,Peripatetic Nurse RGN/RMN - Care Home,46
6,Account Manager,44
7,Support Worker,40
8,Caregiver,39
9,Care Assistant,37


Salary Summary


{'total_jobs': 25000,
 'jobs_with_salary_min': 0,
 'jobs_with_salary_max': 0,
 'jobs_with_salary_text': 0}

In [95]:
#Adaptive Learning from Feedback

adaptive_df = ranked_jobs.copy()

print(adaptive_df.shape)
adaptive_df.head()

(69, 24)


,job_id,title,company,location,country,description,job_url,date_posted,salary_min,salary_max,...,semantic_similarity,violates_dealbreaker,combined_job_text,skill_match_score,matched_skills,role_match_score,location_match_score,semantic_score_norm,final_score,explanation
0,17783,Data Analytics Engineer,NetPace,Remote,us,"Data Analytics Engineer - Remote Pleasanton, C...",https://www.dice.com/jobs/detail/8c815e3146978...,2021-09-01T01:12:14Z,NaN,NaN,...,0.748257,False,Data Analytics Engineer Data Analytics Enginee...,0.176471,"[databricks, python, sql]",1.0,1.0,0.915235,0.751735,"Matched skills: databricks, python, sql | Matc..."
1,16629,DATA SCIENTIST - INTERMEDIATE,"Judge Group, Inc.","Ladue, MO",us,"Location: Ladue, MO Description: Our client is...",https://www.dice.com/jobs/detail/edccbdb22f0a2...,2021-09-01T00:48:47Z,NaN,NaN,...,0.763356,False,"DATA SCIENTIST - INTERMEDIATE Location: Ladue,...",0.117647,"[python, sql]",1.0,0.0,1.000000,0.679412,"Matched skills: python, sql | Matches one of y..."
2,23676,Junior Data Scientist,Michael Page,"Slough,Slough",uk,"My client, a global leader in the automotive s...",https://www.britishjobs.co.uk/job/121000000000...,2021-09-01T04:08:30Z,NaN,NaN,...,0.732668,False,"Junior Data Scientist My client, a global lead...",0.294118,"[documentation, python, r, sql, visualization]",1.0,0.0,0.827720,0.637390,"Matched skills: documentation, python, r, sql,..."
3,23679,Data Scientist,Michael Page,"Slough,Slough",uk,"My client, a global leader in the automotive s...",https://www.britishjobs.co.uk/job/121000000000...,2021-09-01T04:08:30Z,NaN,NaN,...,0.719323,False,"Data Scientist My client, a global leader in t...",0.294118,"[documentation, python, r, sql, visualization]",1.0,0.0,0.752799,0.599929,"Matched skills: documentation, python, r, sql,..."
4,12265,Graduate Data Analyst,ITECCO,"Brighton , East Sussex",uk,My client is seeking a Junior Data Analyst to ...,https://www.reed.co.uk/jobs/graduate-data-anal...,2021-09-01T00:07:10.652Z,NaN,NaN,...,0.712578,False,Graduate Data Analyst My client is seeking a J...,0.235294,"[excel, python, r, sql]",1.0,0.0,0.714931,0.566289,"Matched skills: excel, python, r, sql | Matche..."


In [96]:
def simulate_feedback(row):

    if (
        row["skill_match_score"] >= 0.25 and
        row["role_match_score"] >= 1.0
    ):
        return "accept"

    elif (
        row["skill_match_score"] <= 0.10 and
        row["role_match_score"] == 0
    ):
        return "reject"

    else:
        return "skip"

In [97]:
adaptive_df["feedback"] = adaptive_df.apply(
    simulate_feedback,
    axis=1
)

adaptive_df["feedback"].value_counts()

feedback
skip      50
reject    15
accept     4
Name: count, dtype: int64

In [98]:
adaptive_df["original_rank"] = (
    np.arange(len(adaptive_df)) + 1
)

accepted_jobs = adaptive_df[
    adaptive_df["feedback"] == "accept"
]

baseline_rank = accepted_jobs["original_rank"].mean()

print("Baseline average accepted-job rank:")
print(round(baseline_rank,2))

Baseline average accepted-job rank:
5.75


In [99]:
weights = {
    "semantic": 0.50,
    "skill": 0.25,
    "role": 0.15,
    "location": 0.10
}

weights

{'semantic': 0.5, 'skill': 0.25, 'role': 0.15, 'location': 0.1}

In [100]:
accepted = adaptive_df[
    adaptive_df["feedback"] == "accept"
]

rejected = adaptive_df[
    adaptive_df["feedback"] == "reject"
]

In [101]:
skill_diff = (
    accepted["skill_match_score"].mean()
    - rejected["skill_match_score"].mean()
)

role_diff = (
    accepted["role_match_score"].mean()
    - rejected["role_match_score"].mean()
)

location_diff = (
    accepted["location_match_score"].mean()
    - rejected["location_match_score"].mean()
)

In [102]:
weights["skill"] += 0.50 * skill_diff
weights["role"] += 0.50 * role_diff
weights["location"] += 0.50 * location_diff

In [103]:
total = sum(weights.values())

weights = {
    k: v/total
    for k,v in weights.items()
}

weights

{'semantic': 0.3073214823742091,
 'skill': 0.23169629406447725,
 'role': 0.3995179270864718,
 'location': 0.06146429647484182}

In [104]:
adaptive_df["adaptive_score"] = (
    weights["semantic"] * adaptive_df["semantic_score_norm"]
    +
    weights["skill"] * adaptive_df["skill_match_score"]
    +
    weights["role"] * adaptive_df["role_match_score"]
    +
    weights["location"] * adaptive_df["location_match_score"]
)

In [105]:
adaptive_df = (
    adaptive_df
    .sort_values(
        "adaptive_score",
        ascending=False
    )
    .reset_index(drop=True)
)

adaptive_df["adaptive_rank"] = (
    np.arange(len(adaptive_df)) + 1
)

In [106]:
accepted_after = adaptive_df[
    adaptive_df["feedback"] == "accept"
]

adaptive_rank = accepted_after[
    "adaptive_rank"
].mean()

print("Before:", round(baseline_rank,2))
print("After:", round(adaptive_rank,2))

Before: 5.75
After: 5.5


In [107]:
# Second adaptive-learning iteration

round2_df = adaptive_df.copy()

# Simulate feedback again on the newly re-ranked list
round2_df["feedback_round2"] = round2_df.apply(simulate_feedback, axis=1)

round2_df["round2_rank_before"] = np.arange(len(round2_df)) + 1

accepted_round2 = round2_df[round2_df["feedback_round2"] == "accept"]
rejected_round2 = round2_df[round2_df["feedback_round2"] == "reject"]

round2_baseline_rank = accepted_round2["round2_rank_before"].mean()

# Compute feature differences between accepted and rejected jobs
skill_diff2 = (
    accepted_round2["skill_match_score"].mean()
    - rejected_round2["skill_match_score"].mean()
)

role_diff2 = (
    accepted_round2["role_match_score"].mean()
    - rejected_round2["role_match_score"].mean()
)

location_diff2 = (
    accepted_round2["location_match_score"].mean()
    - rejected_round2["location_match_score"].mean()
)

# Update weights again
weights["skill"] += 0.50 * skill_diff2
weights["role"] += 0.50 * role_diff2
weights["location"] += 0.50 * location_diff2

# Normalize weights
total = sum(weights.values())
weights = {k: v / total for k, v in weights.items()}

print("Updated weights after Round 2:")
print(weights)

# Recompute adaptive score
round2_df["adaptive_score_round2"] = (
    weights["semantic"] * round2_df["semantic_score_norm"] +
    weights["skill"] * round2_df["skill_match_score"] +
    weights["role"] * round2_df["role_match_score"] +
    weights["location"] * round2_df["location_match_score"]
)

# Re-rank again
round2_df = (
    round2_df
    .sort_values("adaptive_score_round2", ascending=False)
    .reset_index(drop=True)
)

round2_df["round2_rank_after"] = np.arange(len(round2_df)) + 1

accepted_after_round2 = round2_df[round2_df["feedback_round2"] == "accept"]

round2_after_rank = accepted_after_round2["round2_rank_after"].mean()

print("Round 2 Before:", round(round2_baseline_rank, 2))
print("Round 2 After:", round(round2_after_rank, 2))

Updated weights after Round 2:
{'semantic': 0.18889298705736265, 'skill': 0.22044604998238435, 'role': 0.5528823655487806, 'location': 0.037778597411472534}
Round 2 Before: 5.5
Round 2 After: 5.25


In [108]:
adaptive_results = pd.DataFrame([
    {
        "round": 0,
        "avg_accepted_rank": baseline_rank,
        "description": "Initial ranking before feedback"
    },
    {
        "round": 1,
        "avg_accepted_rank": adaptive_rank,
        "description": "After first feedback update"
    },
    {
        "round": 2,
        "avg_accepted_rank": round2_after_rank,
        "description": "After second feedback update"
    }
])

adaptive_results

,round,avg_accepted_rank,description
0,0,5.75,Initial ranking before feedback
1,1,5.50,After first feedback update
2,2,5.25,After second feedback update


In [109]:
print(weights)
print("Round 2 Before:", round2_baseline_rank)
print("Round 2 After:", round2_after_rank)

{'semantic': 0.18889298705736265, 'skill': 0.22044604998238435, 'role': 0.5528823655487806, 'location': 0.037778597411472534}
Round 2 Before: 5.5
Round 2 After: 5.25


# Live Job API Ingestion

To satisfy the real-time ingestion requirement, JobPilot integrates
with the Adzuna API to retrieve current job postings.

In [110]:
import sys
print(sys.executable)
print(sys.version)

/Library/Developer/CommandLineTools/usr/bin/python3
3.9.6 (default, May  7 2023, 23:32:45) 
[Clang 14.0.3 (clang-1403.0.22.14.1)]


In [111]:
import sys

!{sys.executable} -m pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [112]:
import os
import requests
import pandas as pd
import hashlib
import re
from dotenv import load_dotenv
import os

load_dotenv("file.env")

ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID")
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")
print("App ID loaded:", ADZUNA_APP_ID is not None)
print("App Key loaded:", ADZUNA_APP_KEY is not None)
def normalize_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def make_dedup_key(title, company, location, description):
    raw = "||".join([
        normalize_text(title),
        normalize_text(company),
        normalize_text(location),
        normalize_text(description)
    ])
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

def fetch_adzuna_jobs(
    query="data analyst",
    country="us",
    results_per_page=50,
    page=1
):
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"

    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "what": query,
        "results_per_page": results_per_page,
        "content-type": "application/json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    return response.json()

def standardize_adzuna_results(api_response):
    rows = []

    for job in api_response.get("results", []):
        title = job.get("title", "")
        company = job.get("company", {}).get("display_name", "")
        location = job.get("location", {}).get("display_name", "")
        description = job.get("description", "")
        job_url = job.get("redirect_url", "")
        date_posted = job.get("created", "")
        salary_min = job.get("salary_min", None)
        salary_max = job.get("salary_max", None)

        rows.append({
            "title": title,
            "company": company,
            "location": location,
            "country": "us",
            "description": description,
            "job_url": job_url,
            "date_posted": date_posted,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "salary_text": "",
            "source": "Adzuna Live API",
            "dedup_key": make_dedup_key(title, company, location, description)
        })

    return pd.DataFrame(rows)

# Fetch live jobs from multiple queries
queries = ["data analyst", "business analyst", "analytics engineer", "data scientist"]

live_dfs = []

for q in queries:
    print(f"Fetching live jobs for: {q}")
    response = fetch_adzuna_jobs(query=q, results_per_page=50)
    live_dfs.append(standardize_adzuna_results(response))

live_jobs_df = pd.concat(live_dfs, ignore_index=True)

before_dedup = len(live_jobs_df)

live_jobs_df = (
    live_jobs_df
    .drop_duplicates(subset=["dedup_key"])
    .reset_index(drop=True)
)

after_dedup = len(live_jobs_df)

print("Live jobs before dedup:", before_dedup)
print("Live jobs after dedup:", after_dedup)

live_jobs_df.head()

App ID loaded: True
App Key loaded: True
Fetching live jobs for: data analyst
Fetching live jobs for: business analyst
Fetching live jobs for: analytics engineer
Fetching live jobs for: data scientist
Live jobs before dedup: 200
Live jobs after dedup: 200


,title,company,location,country,description,job_url,date_posted,salary_min,salary_max,salary_text,source,dedup_key
0,EMPI Data Integrity Analyst - Night Shift,RWJBarnabas Health Corporate Services,"Oceanport, Monmouth County",us,Job Title: EMPI Data Integrity Analyst Locatio...,https://www.adzuna.com/land/ad/5750661956?se=A...,2026-06-03T09:34:39Z,84982.83,84982.83,,Adzuna Live API,20eb88143eff0d6634dce3450dacd78d
1,Registered Nurse (RN) Clinical Data Analyst Ca...,Manatee Memorial Hospital,"Samoset, Manatee County",us,Responsibilities About Manatee Memorial Hospit...,https://www.adzuna.com/land/ad/5743541667?se=A...,2026-05-28T09:21:50Z,114207.58,114207.58,,Adzuna Live API,d5a5ab07455922d79b13d8123aa7b8d1
2,Data Analyst,Uline,"Pleasant Prairie, Kenosha County",us,Data Analyst Corporate Headquarters 12575 Ulin...,https://www.adzuna.com/details/5753065127?utm_...,2026-06-05T06:10:22Z,87959.53,87959.53,,Adzuna Live API,cfdeca839b0e112ef915377c68be1431
3,Data Analyst,"NOVATANG PIONEER TECH, LLC",US,us,We are seeking a highly motivated and detail-o...,https://www.adzuna.com/details/5753065556?utm_...,2026-06-05T06:10:26Z,61270.58,61270.58,,Adzuna Live API,f887d88e00d23d8a90da87e8c02f0de4
4,Data Analyst,"AMERICAN INDUSTRIAL SYSTEMS, INC.","Huntington, Orange County",us,"AMERICAN INDUSTRIAL SYSTEMS, INC. seeks one fu...",https://www.adzuna.com/details/5753062376?utm_...,2026-06-05T06:09:53Z,85983.67,85983.67,,Adzuna Live API,de5c6de5cd251e1fd202d223372c725d


In [113]:
import os

os.makedirs("data", exist_ok=True)

live_jobs_df.to_csv(
    "data/live_jobs_snapshot.csv",
    index=False
)

print("Saved live job snapshot to data/live_jobs_snapshot.csv")

Saved live job snapshot to data/live_jobs_snapshot.csv
